# When is Amazon's biggest day?

_Ten years of European card spending at Amazon, and a few things it tells us that you probably didn't expect._

This notebook reproduces every chart and number in the blog post end-to-end, pulling daily card-transaction data from the [Massive API](https://massive.com) and building up each figure in turn.

You'll need to create a `.env` file with your API key. See `.env.example`

Most of the data comes from the Merchant Aggregates API

https://massive.com/docs/rest/alternative/consumer-spending/merchant-aggregates

Set `MASSIVE_API_KEY` in your environment (or a `.env` file next to this notebook) and run the cells top-to-bottom. The whole thing takes a couple of minutes to fetch a decade of data.

## Setup

In [1]:
import asyncio
import calendar
import datetime as dt
import os

import httpx
import plotly.graph_objects as go
import polars as pl
from dotenv import load_dotenv

load_dotenv(override=True)

API_KEY = os.environ["MASSIVE_API_KEY"]
BASE = "https://api.massive.com"

The Massive API paginates everything via a `next_url` cursor that doesn't carry the API key forward, so we write one tiny async helper and reuse it for every endpoint. Making it `async` lets us fire off several years of data in parallel later.

In [2]:
async def get_all(client: httpx.AsyncClient, endpoint: str, **params) -> list[dict]:
    """Pull every page of a paginated Massive endpoint."""
    params["apiKey"] = API_KEY
    url, rows = f"{BASE}{endpoint}", []
    while url:
        r = await client.get(url, params=params, timeout=60)
        r.raise_for_status()
        payload = r.json()
        rows.extend(payload.get("results", []))
        url = payload.get("next_url")
        params = {"apiKey": API_KEY} | params  # next_url carries its own filters
    return rows

## Finding the Amazon retail merchants

"Amazon" on a card statement is not one merchant — it's a family of them. There are over a hundred merchant strings that roll up to the `AMZN US` ticker: `amazon uk`, `amazon germany`, Whole Foods, Twitch, Ring, Kindle, Audible, AWS, Amazon Pay. For a retail-seasonality story we want the `amazon.xx` shopping sites only.

Amazon Pay in particular is a payment processor and shows up as _positive_ card flow — including it would make the numbers wrong in a way that's easy to miss.

In [3]:
async with httpx.AsyncClient() as client:
    hierarchy = await get_all(
        client,
        "/consumer-spending/eu/v1/merchant-hierarchy",
        listing_status="public",
        limit=50000,
    )

# Amazon retail brands only: exclude subscriptions, AWS, Pay, Whole Foods, etc.
SKIP = (
    "amazon pay",
    "amazon webservices",
    "amazon prime",
    "amazon audible",
    "amazon kindle",
    "amazon logistics",
    "amazon market services",
    "amazon sarl",
)
amzn_retail = sorted(
    {
        m["lookup_name"]
        for m in hierarchy
        if (m.get("parent_ticker") or "") == "AMZN US"
        and (m["lookup_name"] or "").startswith("amazon")
        and not any(s in m["lookup_name"] for s in SKIP)
    }
)
amzn_retail

['amazon',
 'amazon eu',
 'amazon france',
 'amazon germany',
 'amazon italy',
 'amazon luxembourg',
 'amazon spain',
 'amazon uk']

## Pulling ten years of daily spending

Pagination through a whole year is sequential — page N blocks until page N-1 arrives. The trick is to split by month, so all 11 x 12 = 132 month-queries fan out concurrently. A semaphore caps concurrency; one `httpx.AsyncClient` pools connections across the lot. Without this concurrency the data download takes a long time!

In [4]:
CONCURRENCY = 20


async def fetch_month(client, sem, year, month):
    async with sem:
        last = calendar.monthrange(year, month)[1]
        return await get_all(
            client,
            "/consumer-spending/eu/v1/merchant-aggregates",
            **{
                "transaction_date.gte": f"{year}-{month:02d}-01",
                "transaction_date.lte": f"{year}-{month:02d}-{last:02d}",
                "name.any_of": ",".join(amzn_retail),
                "limit": 5000,
            },
        )


years = list(range(2016, 2027))
sem = asyncio.Semaphore(CONCURRENCY)
limits = httpx.Limits(max_connections=CONCURRENCY + 2, max_keepalive_connections=5)

async with httpx.AsyncClient(timeout=90, limits=limits) as client:
    tasks = [fetch_month(client, sem, y, m) for y in years for m in range(1, 13)]
    chunks = await asyncio.gather(*tasks)

rows = [r for chunk in chunks for r in chunk]
print(f"total: {len(rows):,} rows across {len(tasks)} month-queries")

total: 203,392 rows across 132 month-queries


A few gotchas before we aggregate:

- **Spending is signed negative.** Card debits are outflows. Flip the sign, or every chart will be upside down.
- **The panel grows over time.** The number of cards in the panel in 2016 is a small fraction of what it is today. For cross-year comparisons we'll normalise by `twenty_eight_day_rolling_total_accounts`, which is the trailing-28-day count of active cards — a property of the panel rather than of Amazon. We also keep `total_accounts` (accounts that actually spent at Amazon that day) for a per-active-account view.

In [5]:
raw = pl.DataFrame(rows).with_columns(
    pl.col("transaction_date").cast(pl.Date),
    pl.col("total_spend").neg().alias("spend"),  # debits → positive
)

# Panel size is a panel-wide attribute stored on every row; take the max per day.
panel = raw.group_by("transaction_date").agg(
    pl.col("twenty_eight_day_rolling_total_accounts").max().alias("panel_28d_accts")
)

daily = (
    raw.group_by("transaction_date")
    .agg(pl.col("spend").sum())
    .join(panel, on="transaction_date")
    .sort("transaction_date")
    .rename({"transaction_date": "date"})
    .with_columns(
        (pl.col("spend") / pl.col("panel_28d_accts")).alias("spend_per_acct"),
        (pl.col("spend") / 1e6).alias("spend_m"),
    )
)
print(daily.shape)
daily.head(3)

(3735, 5)


date,spend,panel_28d_accts,spend_per_acct,spend_m
date,f64,i64,f64,f64
2016-01-01,34194.11,72633,0.470779,0.034194
2016-01-02,202673.86,91598,2.212645,0.202674
2016-01-03,151983.29,109762,1.384662,0.151983


## Plot style

One shared Plotly template so every chart looks like a sibling of the others.

In [6]:
PALETTE = {
    "primary": "#0088A9",
    "accent": "#FF6B35",
    "pink": "#C2185B",
    "green": "#2E7D32",
    "grey": "#B0B0B0",
    "gold": "#FFB300",
}

TEMPLATE = go.layout.Template(
    layout=go.Layout(
        font=dict(family="Inter, Helvetica, Arial, sans-serif", size=13, color="#222"),
        plot_bgcolor="white",
        paper_bgcolor="white",
        xaxis=dict(
            showgrid=False,
            zeroline=False,
            showline=True,
            linecolor="#999",
            ticks="outside",
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor="#eee",
            zeroline=False,
            showline=False,
            ticks="outside",
        ),
        margin=dict(l=80, r=30, t=60, b=50),
        colorway=[
            PALETTE["primary"],
            PALETTE["accent"],
            PALETTE["green"],
            PALETTE["pink"],
            PALETTE["gold"],
        ],
    )
)

## 1. Ten years, one chart

The hook. Daily European card spending at Amazon retail, start of 2016 to the most recent data. The five biggest days are split between Prime Day and Black Friday week, and they're all in the last two years — which already hints that "biggest day ever" is a trick question we'll come back to.

In [7]:
top5 = daily.sort("spend_m", descending=True).head(5)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=daily["date"].to_list(),
        y=daily["spend_m"].to_list(),
        mode="lines",
        line=dict(color=PALETTE["primary"], width=0.8),
        showlegend=False,
    )
)
fig.add_trace(
    go.Scatter(
        x=top5["date"].to_list(),
        y=top5["spend_m"].to_list(),
        mode="markers",
        marker=dict(color=PALETTE["accent"], size=10),
        name="Top 5 days",
        showlegend=False,
        hovertemplate="%{x}<br>£%{y:.2f}M<extra></extra>",
    )
)
fig.update_layout(
    template=TEMPLATE,
    title="Ten years of European card spending at Amazon (£M per day)",
    yaxis_title="£M per day",
    height=420,
)
fig.show()

## 2. The shape of a single year

Before picking a winner, look at 2024 day-by-day. Three distinct peaks stand out, followed by the most dramatic pattern in the data that nobody talks about: the Christmas cliff.

- **Prime Day** — a sharp two-day spike in mid-July, about twice the surrounding baseline.
- **Black Friday / Cyber Week** — not a spike but a plateau, roughly 50% above baseline, running from late November through to mid-December.
- **The Christmas cliff** — spending rises into the week of Dec 18–20, then collapses as shipping deadlines close. On Christmas Eve 2024, Amazon card spend was about one-sixth of its peak eight days earlier. Boxing Day recovers about half of it.

In [8]:
year = 2024
sub = daily.filter(pl.col("date").dt.year() == year).sort("date")

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=sub["date"].to_list(),
        y=sub["spend_m"].to_list(),
        marker_color=PALETTE["grey"],
        marker_line_width=0,
        hovertemplate="%{x}<br>£%{y:.2f}M<extra></extra>",
    )
)


def y_at(d):
    return sub.filter(pl.col("date") == d).select("spend_m").item()


annots = [
    ("Prime Day (Jul 16–17)", dt.date(2024, 7, 16), PALETTE["accent"]),
    ("Black Friday / Cyber Week", dt.date(2024, 11, 29), PALETTE["accent"]),
    ("Shipping cutoff (Dec 20ish)", dt.date(2024, 12, 20), PALETTE["accent"]),
    ("Christmas Day", dt.date(2024, 12, 25), PALETTE["green"]),
]
fig.update_layout(
    template=TEMPLATE,
    title=f"{year}: three distinct peaks, then the Christmas Day cliff",
    yaxis_title="£M per day",
    height=400,
    showlegend=False,
    bargap=0,
    annotations=[
        dict(
            x=d,
            y=y_at(d),
            xref="x",
            yref="y",
            text=lbl,
            showarrow=True,
            arrowhead=2,
            ax=0,
            ay=-35,
            font=dict(size=11, color=c),
        )
        for lbl, d, c in annots
    ],
)
fig.show()

## 3. Prime Day vs Black Friday: the day itself

Which of the first two is actually bigger? One natural way to ask: on the peak day itself, how much more does Europe spend than on a nearby ordinary day?

We compute the **lift**: peak-day spend divided by the average of a ±14-day window around it, with the event itself taken out of the baseline.

In [9]:
def lift(event: list[dt.date], window: int = 14) -> float:
    """Peak day of an event ÷ average of a ±14-day window around it,
    with the event itself taken out of the baseline."""
    start = min(event) - dt.timedelta(days=window)
    end = max(event) + dt.timedelta(days=window)
    base = (
        daily.filter(pl.col("date").is_between(start, end))
        .filter(~pl.col("date").is_in(event))["spend"]
        .mean()
    )
    peak = daily.filter(pl.col("date").is_in(event))["spend"].max()
    return peak / base


print(
    f"Prime Day 2024    lift: {lift([dt.date(2024, 7, 16), dt.date(2024, 7, 17)]):.2f}"
)
print(f"Black Friday 2024 lift: {lift([dt.date(2024, 11, 29)]):.2f}")

Prime Day 2024    lift: 2.31
Black Friday 2024 lift: 1.41


Now repeat for every Prime Day and Black Friday since 2016.

Event dates: Prime Day is mostly a two-day July event (one day in 2016–17, stretched to four days in 2025, and delayed to October 2020 for COVID). Black Friday is the Friday after the fourth Thursday of November.

In [10]:
PRIME_DAYS = {
    2016: [dt.date(2016, 7, 12)],
    2017: [dt.date(2017, 7, 11)],
    2018: [dt.date(2018, 7, 16), dt.date(2018, 7, 17)],
    2019: [dt.date(2019, 7, 15), dt.date(2019, 7, 16)],
    2020: [dt.date(2020, 10, 13), dt.date(2020, 10, 14)],  # COVID delay
    2021: [dt.date(2021, 6, 21), dt.date(2021, 6, 22)],
    2022: [dt.date(2022, 7, 12), dt.date(2022, 7, 13)],
    2023: [dt.date(2023, 7, 11), dt.date(2023, 7, 12)],
    2024: [dt.date(2024, 7, 16), dt.date(2024, 7, 17)],
    2025: [
        dt.date(2025, 7, 8),
        dt.date(2025, 7, 9),
        dt.date(2025, 7, 10),
        dt.date(2025, 7, 11),
    ],
}


def black_friday(year: int) -> dt.date:
    """Friday after the 4th Thursday of November."""
    d = dt.date(year, 11, 1)
    first_thu = d + dt.timedelta(days=(3 - d.weekday()) % 7)
    return first_thu + dt.timedelta(days=22)  # +3 weeks +1 day


years = list(range(2016, 2026))
pd_lifts = [lift(PRIME_DAYS[y]) for y in years]
bf_lifts = [lift([black_friday(y)]) for y in years]

for y, p, b in zip(years, pd_lifts, bf_lifts):
    print(f"  {y}   PD {p:.2f}x   BF {b:.2f}x")

  2016   PD 2.37x   BF 1.36x
  2017   PD 2.49x   BF 1.38x
  2018   PD 2.44x   BF 1.90x
  2019   PD 2.81x   BF 1.94x
  2020   PD 2.17x   BF 1.60x
  2021   PD 2.38x   BF 1.52x
  2022   PD 2.09x   BF 1.50x
  2023   PD 2.17x   BF 1.52x
  2024   PD 2.31x   BF 1.41x
  2025   PD 2.02x   BF 1.37x


In [11]:
fig = go.Figure()
fig.add_trace(
    go.Bar(x=years, y=pd_lifts, name="Prime Day", marker_color=PALETTE["accent"])
)
fig.add_trace(
    go.Bar(x=years, y=bf_lifts, name="Black Friday", marker_color=PALETTE["primary"])
)
fig.add_hline(y=1.0, line=dict(color="#999", dash="dot", width=1))
fig.update_layout(
    template=TEMPLATE,
    title="How much bigger than normal is each day? (peak day ÷ ±14-day baseline)",
    yaxis_title="Lift vs baseline (x)",
    barmode="group",
    height=420,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)
fig.show()

Prime Day wins on peak-day lift every year. The best Prime Days (2019, 2023, 2024) hit 2.3–2.8x a normal day. Black Friday peaked at 2.0x in 2018 and hasn't matched that since.

Two wrinkles before declaring a winner:

1. **Prime Day's per-day punch is fading.** As Amazon stretches the window (one day → two → four, plus an October "Big Deal Days"), the peak intensity has fallen. In 2019 the peak Prime Day hit nearly 2.9x; in 2025, now four days long, it managed exactly 2.0x.
2. **Black Friday isn't a single day** — it kicks off a seven-to-ten-day elevated period that Prime Day doesn't match. Integrate the area under each peak and Cyber Week wins every year.

## 4. The answer depends on where you live

Aggregating Europe into one number hides something more interesting: the countries behave very differently on Prime Day, and very similarly on Black Friday.

We need the same daily series split by `user_country`. Recompute the retail filter directly off the raw rows and group by date and country.

In [12]:
by_country = (
    raw.filter(pl.col("user_country").is_in(["UK", "DE", "FR", "IT", "ES"]))
    .group_by(["transaction_date", "user_country"])
    .agg(pl.col("spend").sum())
    .rename({"transaction_date": "date"})
    .sort(["user_country", "date"])
)


def country_lift(country: str, event: list[dt.date], window: int = 14) -> float:
    sub = by_country.filter(pl.col("user_country") == country)
    start = min(event) - dt.timedelta(days=window)
    end = max(event) + dt.timedelta(days=window)
    base = (
        sub.filter(pl.col("date").is_between(start, end))
        .filter(~pl.col("date").is_in(event))["spend"]
        .mean()
    )
    peak = sub.filter(pl.col("date").is_in(event))["spend"].max()
    return peak / base


countries = ["IT", "ES", "DE", "UK", "FR"]
pd_event = PRIME_DAYS[2024]
bf_event = [
    black_friday(2024),
    black_friday(2024) + dt.timedelta(days=3),
]  # Cyber Monday

pd_by_country = [country_lift(c, pd_event) for c in countries]
bf_by_country = [country_lift(c, bf_event) for c in countries]

for c, p, b in zip(countries, pd_by_country, bf_by_country):
    print(f"  {c}   PD {p:.2f}x   BF {b:.2f}x")

  IT   PD 4.83x   BF 1.60x
  ES   PD 3.00x   BF 1.95x
  DE   PD 2.26x   BF 1.77x
  UK   PD 1.87x   BF 1.33x
  FR   PD 1.52x   BF 1.59x


In [13]:
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=countries,
        y=pd_by_country,
        name="Prime Day 2024",
        marker_color=PALETTE["accent"],
        text=[f"{v:.1f}x" for v in pd_by_country],
        textposition="outside",
    )
)
fig.add_trace(
    go.Bar(
        x=countries,
        y=bf_by_country,
        name="Black Friday 2024",
        marker_color=PALETTE["primary"],
        text=[f"{v:.1f}x" for v in bf_by_country],
        textposition="outside",
    )
)
fig.add_hline(y=1.0, line=dict(color="#999", dash="dot", width=1))
fig.update_layout(
    template=TEMPLATE,
    title="Peak-day lift by country: Italy and Spain do Prime Day, everyone else is more uniform",
    yaxis_title="Peak day ÷ ±14-day baseline",
    barmode="group",
    height=440,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    yaxis=dict(range=[0, max(pd_by_country) * 1.18]),
)
fig.show()

Italy goes berserk for Prime Day — card spend on Jul 16 2024 was almost 5x its surrounding July baseline. Spain is second, Germany third. The UK and France barely notice.

Black Friday is the opposite: every country sits in a narrow 1.3–2.0x band. That's not because Europeans don't care about Black Friday — it's because Black Friday is a plateau, not a single-day manufactured event. You can only concentrate so much of a week-long sale onto one day.

_Why_ the Italy/Spain tilt? One hypothesis: Prime Day is a brand event, not a price event, and it works best where Amazon is the default reference point for online retail (more true in Italy and Spain than the UK or France). The shapes in the chart are the fact; the reason is still a guess.

## 5. So what is the biggest day ever, really?

Now the twist.

The raw time series at the top of the notebook shows the biggest days clustered in the last two years. That looks like growth (and Amazon has been growing), but it's also a trick of the panel. In January 2016 the panel tracked tens of thousands of active cards across Europe; today it's close to a million. Headline daily takings grew roughly 8x over a decade, and most of that is simply us tracking more cards.

The clean fix is to divide by `panel_28d_accts` — the trailing-28-day count of active cards in the panel. That puts 2016 and 2024 on the same scale. Re-rank the top 10 days of the decade:

In [14]:
top10 = daily.sort("spend_per_acct", descending=True).head(10)


def label_date(d: dt.date) -> str:
    for y, days in PRIME_DAYS.items():
        if d in days:
            tag = " (Oct, COVID)" if y == 2020 else ""
            return f"Prime Day {y}{tag} ({d:%b %d})"
    if d == black_friday(d.year):
        return f"Black Friday {d.year} ({d:%b %d})"
    if d == black_friday(d.year) + dt.timedelta(days=3):
        return f"Cyber Monday {d.year} ({d:%b %d})"
    return f"{d:%b %d, %Y}"


labels = [label_date(d) for d in top10["date"].to_list()]
vals = top10["spend_per_acct"].to_list()
colors = [
    PALETTE["accent"] if "Black Friday 2020" in l else PALETTE["primary"]
    for l in labels
]

fig = go.Figure(
    go.Bar(
        x=vals[::-1],
        y=labels[::-1],
        orientation="h",
        marker_color=colors[::-1],
        text=[f"£{v:.2f}" for v in vals[::-1]],
        textposition="outside",
        hovertemplate="%{y}<br>£%{x:.2f} per active account<extra></extra>",
    )
)
fig.update_layout(
    template=TEMPLATE,
    title="Top 10 days for Amazon-Europe spend — per active account in the panel",
    xaxis_title="Spend per active account (£)",
    height=460,
    showlegend=False,
    margin=dict(l=260, r=60, t=60, b=50),
    xaxis=dict(range=[0, max(vals) * 1.15]),
)
fig.show()

The biggest day in ten years of European card data is **Black Friday 2020**.

2020 was the year Amazon pushed Prime Day to October because of the pandemic. It was the year lockdown pushed home delivery from a convenience into a necessity. The bizarre October Prime Day was bigger (per active account) than any July Prime Day before 2024. And late-November Black Friday 2020 was bigger than all of them. Four of the top ten days in the decade sit inside a six-week window in late 2020.

You wouldn't spot this from the raw spend chart at the top, because 2020 looks modest next to 2024–25. Once you control for panel size the reality flips: 2020 was off the charts, and the post-2020 years have been slowly catching up with what lockdown did.

## A short note on the data

Three things that will trip you up if you try to redo this analysis with a different retailer:

- **Spending is signed negative.** Card debits are outflows. Flip the sign before plotting.
- **"Amazon" isn't one merchant.** The merchant hierarchy has over a hundred strings rolling up to `AMZN US`. For a retail-seasonality story you want the `amazon.xx` shopping sites only — not Pay, AWS, Whole Foods, Twitch, Ring, Audible, or Kindle. Amazon Pay in particular is a payment processor and shows up as positive card flow.
- **The panel grows over time.** Any chart of "Amazon spend at time t" is partly a chart of "how many cards we were tracking at time t". For cross-year comparisons, divide by `twenty_eight_day_rolling_total_accounts`.

None of this is rocket science, but getting it right is the difference between a chart that tells the truth and one that tells a plausible-looking lie. Swap `AMZN US` for any other ticker in the merchant hierarchy to repeat the exercise for a different retailer.